### This notebook extract newly generated molecules after the DORAnet run: 

In [16]:
import os
import glob
import pandas as pd
from pathlib import Path
from collections import defaultdict
from rdkit import RDLogger
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Draw
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import io

RDLogger.DisableLog('rdApp.*')

In [17]:
fileNamePrefix = "drugCentralDORAnetGeneratedMolecules"

### Discover All Starter Directories

In [18]:
doranetOutputDir = "doranet_output"

helpersToExclude = {
    'O', 'O=O', '[H][H]', 'O=C=O', 'C=O', '[C-]#[O+]', 'Br', '[Br][Br]',
    'CO', 'C=C', 'O=S(O)O', 'N', 'O=S(=O)(O)O', 'O=NO', 'N#N',
    'O=[N+]([O-])O', 'NO', 'C#N', 'S', 'O=S=O', 'N#CO', '[H+]', 'OO',
    'Cl', 'I', 'O=C(O)O', 'O=P(O)(O)O', 'O=P(O)(O)OP(=O)(O)O', 'C',
    'CC', 'CC=O', 'CC(=O)O', 'CCC(=O)O'
}

print(f"Output directory: {doranetOutputDir}")
print(f"Number of helpers to exclude: {len(helpersToExclude)}")

Output directory: doranet_output
Number of helpers to exclude: 33


In [19]:
# Automatically find all starter directories (exclude files)
allStarterDirPaths = sorted([
    p for p in glob.glob(os.path.join(doranetOutputDir, "starter_*"))
    if os.path.isdir(p)
])

print(f"Found {len(allStarterDirPaths)} starter directories")

# Find the molecules CSV file inside each directory
discoveredCsvFiles = []
directoriesWithMissingCsv = []

for starterDirPath in allStarterDirPaths:
    dirName = os.path.basename(starterDirPath)

    # Expected CSV file name matches directory name
    expectedCsvPath = os.path.join(starterDirPath, f"{dirName}_molecules.csv")

    if os.path.exists(expectedCsvPath):
        discoveredCsvFiles.append({
            'dirName': dirName,
            'dirPath': starterDirPath,
            'csvPath': expectedCsvPath,
            'starterNum': int(dirName.replace('starter_', ''))
        })
    else:
        # Try to find any molecules CSV in the directory as fallback
        fallbackCsvPaths = glob.glob(os.path.join(starterDirPath, "*_molecules.csv"))
        if fallbackCsvPaths:
            discoveredCsvFiles.append({
                'dirName': dirName,
                'dirPath': starterDirPath,
                'csvPath': fallbackCsvPaths[0],
                'starterNum': int(dirName.replace('starter_', ''))
            })
        else:
            directoriesWithMissingCsv.append(dirName)

# Sort by starter number
discoveredCsvFiles.sort(key=lambda x: x['starterNum'])

print(f"Found {len(discoveredCsvFiles)} CSV files containing DORAnet generated molecules")
if directoriesWithMissingCsv:
    print(f"Missing CSV in {len(directoriesWithMissingCsv)} directories: "
          f"{directoriesWithMissingCsv[:10]}...")

Found 81 starter directories
Found 17 CSV files containing DORAnet generated molecules
Missing CSV in 64 directories: ['starter_00000', 'starter_00001', 'starter_00002', 'starter_00003', 'starter_00004', 'starter_00006', 'starter_00008', 'starter_00009', 'starter_00010', 'starter_00011']...


### Read All CSV Files into a DataFrame

In [20]:
# Read all CSV files and combine into one DataFrame
perStarterDataFrames = []
csvReadErrors = []

for fileInfo in discoveredCsvFiles:
    try:
        singleStarterMolecules = pd.read_csv(fileInfo['csvPath'])

        # Add source starter information
        singleStarterMolecules['SourceStarterNum'] = fileInfo['starterNum']
        singleStarterMolecules['SourceDirectory'] = fileInfo['dirName']

        perStarterDataFrames.append(singleStarterMolecules)

    except Exception as e:
        csvReadErrors.append({
            'dirName': fileInfo['dirName'],
            'csvPath': fileInfo['csvPath'],
            'error': str(e)
        })

# Concatenate all DataFrames
if perStarterDataFrames:
    allMoleculesWithDuplicates = pd.concat(perStarterDataFrames, ignore_index=True)
    print(f"Total rows read (with duplicates and helpers): {len(allMoleculesWithDuplicates)}")
    print(f"Columns: {list(allMoleculesWithDuplicates.columns)}")
    print(f"Unique SMILES (before cleaning): {allMoleculesWithDuplicates['SMILES'].nunique()}")
else:
    print("ERROR: No CSV files could be read!")

if csvReadErrors:
    print(f"\nFailed to read {len(csvReadErrors)} files:")
    for err in csvReadErrors[:5]:
        print(f"  {err['dirName']}: {err['error']}")

allMoleculesWithDuplicates

Total rows read (with duplicates and helpers): 28353
Columns: ['SMILES', 'Is_Starter', 'MolFormula', 'MolWeight', 'NumHeavyAtoms', 'SourceStarterNum', 'SourceDirectory']
Unique SMILES (before cleaning): 27543


,SMILES,Is_Starter,MolFormula,MolWeight,NumHeavyAtoms,SourceStarterNum,SourceDirectory
0,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,False,C10H16N5O13P3,506.9957,31,5,starter_00005
1,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)O)[C@@H]...,False,C10H14N5O7P,347.0631,23,5,starter_00005
2,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)...,False,C27H33N9O15P2,785.1571,53,5,starter_00005
3,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,False,C27H35N9O15P2,787.1728,53,5,starter_00005
4,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,False,C10H15N5O10P2,427.0294,27,5,starter_00005
...,...,...,...,...,...,...,...
28348,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,C51H45F3N18O19P4,1394.2011,95,73,starter_00073
28349,Cn1nnc(-c2ccc(-c3ccc(OP(=O)(OC[C@H]4CN(c5ccc(-...,False,C47H38F3N17O11P2,1135.2364,80,73,starter_00073
28350,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,C21H22FN7O11P2,629.0837,42,73,starter_00073
28351,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,C68H58F4N24O21P4,1746.3095,121,73,starter_00073


### Remove HELPERS and Deduplicate

In [21]:
# Step 1: Remove helpers
allMoleculesWithoutHelpers = allMoleculesWithDuplicates[
    ~allMoleculesWithDuplicates['SMILES'].isin(helpersToExclude)
].copy()

helpersRemovedCount = len(allMoleculesWithDuplicates) - len(allMoleculesWithoutHelpers)
helpersKeptPercent = (len(allMoleculesWithoutHelpers) / len(allMoleculesWithDuplicates)) * 100
print(f"Step 1 - Removed {helpersRemovedCount} helper molecules")
print(f"Remaining molecules: {len(allMoleculesWithoutHelpers)} ({helpersKeptPercent:.2f}%)")

# Step 2: Aggregate source starters per unique SMILES
sourceStarterMapping = (
    allMoleculesWithoutHelpers
    .groupby('SMILES')['SourceStarterNum']
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)
sourceStarterMapping.columns = ['SMILES', 'SourceStarterList']
sourceStarterMapping['NumSourceStarters'] = sourceStarterMapping['SourceStarterList'].apply(len)
sourceStarterMapping['SourceStarters'] = sourceStarterMapping['SourceStarterList'].apply(
    lambda x: ';'.join(map(str, x))
)

# Also aggregate source directories per unique SMILES
sourceDirectoryMapping = (
    allMoleculesWithoutHelpers
    .groupby('SMILES')['SourceDirectory']
    .apply(lambda x: ';'.join(sorted(set(x))))
    .reset_index()
)
sourceDirectoryMapping.columns = ['SMILES', 'SourceDirectories']

# Step 3: Deduplicate - keep first occurrence of each SMILES
uniqueMolecules = allMoleculesWithoutHelpers.drop_duplicates(
    subset='SMILES', keep='first'
).copy()

# Keep only desired columns
uniqueMolecules = uniqueMolecules[['SMILES', 'Is_Starter']].copy()

# Step 4: Merge aggregated source starter and directory information
uniqueMolecules = uniqueMolecules.merge(
    sourceStarterMapping[['SMILES', 'NumSourceStarters', 'SourceStarters']],
    on='SMILES',
    how='left'
)
uniqueMolecules = uniqueMolecules.merge(
    sourceDirectoryMapping,
    on='SMILES',
    how='left'
)

duplicatesRemovedCount = len(allMoleculesWithoutHelpers) - len(uniqueMolecules)
dedupeKeptPercent = (len(uniqueMolecules) / len(allMoleculesWithoutHelpers)) * 100
overallKeptPercent = (len(uniqueMolecules) / len(allMoleculesWithDuplicates)) * 100

print(f"Step 2 - Removed {duplicatesRemovedCount} duplicate molecular SMILES ({dedupeKeptPercent:.2f}%)")
print(f"Final unique molecules: {len(uniqueMolecules)} ({overallKeptPercent:.2f}% of original)")

uniqueMolecules = uniqueMolecules.drop(columns=['NumSourceStarters', 'SourceStarters'])
csvOutputPath = os.path.join(doranetOutputDir, f"{fileNamePrefix}.csv")
uniqueMolecules.to_csv(csvOutputPath, index=False, encoding="utf-8")
print(f"All the newly DORAnet generated molecular SMILES saved to: {os.path.abspath(csvOutputPath)}")
uniqueMolecules

Step 1 - Removed 304 helper molecules
Remaining molecules: 28049 (98.93%)
Step 2 - Removed 528 duplicate molecular SMILES (98.12%)
Final unique molecules: 27521 (97.07% of original)
All the newly DORAnet generated molecular SMILES saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/doranet/scripts/DrugCentral/doranet_output/drugCentralDORAnetGeneratedMolecules.csv


,SMILES,Is_Starter,SourceDirectories
0,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,False,starter_00005;starter_00007;starter_00014;star...
1,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)O)[C@@H]...,False,starter_00005;starter_00007;starter_00014;star...
2,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)...,False,starter_00005;starter_00007;starter_00014;star...
3,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,False,starter_00005;starter_00007;starter_00014;star...
4,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,False,starter_00005;starter_00007;starter_00014;star...
...,...,...,...
27516,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,starter_00073
27517,Cn1nnc(-c2ccc(-c3ccc(OP(=O)(OC[C@H]4CN(c5ccc(-...,False,starter_00073
27518,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,starter_00073
27519,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,starter_00073


In [25]:
# ************************************************************
# Step 1: Remove helpers
# ************************************************************
allMoleculesWithoutHelpers = allMoleculesWithDuplicates[
    ~allMoleculesWithDuplicates['SMILES'].isin(helpersToExclude)
].copy()

helpersRemovedCount = len(allMoleculesWithDuplicates) - len(allMoleculesWithoutHelpers)
helpersKeptPercent = (len(allMoleculesWithoutHelpers) / len(allMoleculesWithDuplicates)) * 100
print(f"Step 1 - Removed {helpersRemovedCount} helper molecules")
print(f"Remaining molecules: {len(allMoleculesWithoutHelpers)} ({helpersKeptPercent:.2f}%)")

# ************************************************************
# Step 2: Aggregate source starters per unique SMILES
# ************************************************************
sourceStarterMapping = (
    allMoleculesWithoutHelpers
    .groupby('SMILES')['SourceStarterNum']
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)
sourceStarterMapping.columns = ['SMILES', 'SourceStarterList']
sourceStarterMapping['NumSourceStarters'] = sourceStarterMapping['SourceStarterList'].apply(len)
sourceStarterMapping['SourceStarters'] = sourceStarterMapping['SourceStarterList'].apply(
    lambda x: ';'.join(map(str, x))
)

sourceDirectoryMapping = (
    allMoleculesWithoutHelpers
    .groupby('SMILES')['SourceDirectory']
    .apply(lambda x: ';'.join(sorted(set(x))))
    .reset_index()
)
sourceDirectoryMapping.columns = ['SMILES', 'SourceDirectories']

# Step 3: Deduplicate
uniqueMolecules = allMoleculesWithoutHelpers.drop_duplicates(
    subset='SMILES', keep='first'
).copy()
uniqueMolecules = uniqueMolecules[['SMILES', 'Is_Starter']].copy()

# ************************************************************
# Step 4: Merge aggregated information
# ************************************************************
uniqueMolecules = uniqueMolecules.merge(
    sourceStarterMapping[['SMILES', 'NumSourceStarters', 'SourceStarters']],
    on='SMILES', how='left'
)
uniqueMolecules = uniqueMolecules.merge(
    sourceDirectoryMapping, on='SMILES', how='left'
)

duplicatesRemovedCount = len(allMoleculesWithoutHelpers) - len(uniqueMolecules)
dedupeKeptPercent = (len(uniqueMolecules) / len(allMoleculesWithoutHelpers)) * 100
overallKeptPercent = (len(uniqueMolecules) / len(allMoleculesWithDuplicates)) * 100

print(f"Step 2 - Removed {duplicatesRemovedCount} duplicate molecular SMILES ({dedupeKeptPercent:.2f}%)")
print(f"Final unique molecules: {len(uniqueMolecules)} ({overallKeptPercent:.2f}% of original)")

uniqueMolecules = uniqueMolecules.drop(columns=['NumSourceStarters', 'SourceStarters'])

# ************************************************************
# Step 5: Build starter -> generated molecules mapping
# ************************************************************

# Separate starters and generated molecules
starterSmiles = uniqueMolecules[uniqueMolecules['Is_Starter'] == True]['SMILES'].tolist()
generatedMolecules = allMoleculesWithoutHelpers[
    allMoleculesWithoutHelpers['Is_Starter'] == False
].copy()

# Build mapping: starter num -> starter SMILES
starterNumToSmiles = (
    allMoleculesWithoutHelpers[allMoleculesWithoutHelpers['Is_Starter'] == True]
    .drop_duplicates(subset='SourceStarterNum')
    .set_index('SourceStarterNum')['SMILES']
    .to_dict()
)

# Build mapping: starter SMILES -> list of unique generated SMILES
starterToGeneratedMap = {}
for starterNum, starterSmi in starterNumToSmiles.items():
    genSmiles = (
        generatedMolecules[generatedMolecules['SourceStarterNum'] == starterNum]['SMILES']
        .unique()
        .tolist()
    )
    if genSmiles:
        starterToGeneratedMap[starterSmi] = genSmiles

print(f"\nStarters with generated molecules: {len(starterToGeneratedMap)}")

# ************************************************************
# Step 6: Create GIF frames
# ************************************************************

def smiles_to_image(smiles, img_size=(250, 250)):
    """Convert SMILES to a PIL image."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        img = Image.new('RGB', img_size, 'white')
        draw = ImageDraw.Draw(img)
        draw.text((10, img_size[1] // 2), "Invalid SMILES", fill='red')
        return img
    return Draw.MolToImage(mol, size=img_size)


def create_frame(starter_entries, frame_width=1600, row_height=300, mol_size=(250, 250)):
    """
    Create one GIF frame showing up to 3 starters, each with up to 2 generated molecules.
    starter_entries: list of tuples (starter_smiles, [gen_smiles_1, gen_smiles_2], start_index)
    """
    frame_height = row_height * len(starter_entries)
    frame = Image.new('RGB', (frame_width, frame_height), 'white')
    draw = ImageDraw.Draw(frame)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
        title_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 16)
    except (IOError, OSError):
        font = ImageFont.load_default()
        title_font = font

    for row_idx, (starter_smi, gen_smiles_list, start_index) in enumerate(starter_entries):
        y_offset = row_idx * row_height
        padding = 20

        # Draw starter molecule on the left
        starter_img = smiles_to_image(starter_smi, mol_size)
        frame.paste(starter_img, (padding, y_offset + padding))
        draw.text(
            (padding, y_offset + padding + mol_size[1] + 5),
            "Starter compound", fill='blue', font=title_font
        )

        # Draw arrow
        arrow_x_start = padding + mol_size[0] + 20
        arrow_x_end = arrow_x_start + 80
        arrow_y = y_offset + padding + mol_size[1] // 2

        draw.line([(arrow_x_start, arrow_y), (arrow_x_end, arrow_y)], fill='black', width=3)
        draw.polygon([
            (arrow_x_end, arrow_y - 10),
            (arrow_x_end + 15, arrow_y),
            (arrow_x_end, arrow_y + 10)
        ], fill='black')

        # Draw up to 2 generated molecules on the right
        gen_x_start = arrow_x_end + 40
        for gen_idx, gen_smi in enumerate(gen_smiles_list[:3]):
            gen_x = gen_x_start + gen_idx * (mol_size[0] + 40)
            gen_img = smiles_to_image(gen_smi, mol_size)
            frame.paste(gen_img, (gen_x, y_offset + padding))
            draw.text(
                (gen_x, y_offset + padding + mol_size[1] + 5),
                f"DORAnet Generated {start_index + gen_idx + 1}", fill='green', font=font
            )

        # Draw separator line between rows
        if row_idx < len(starter_entries) - 1:
            sep_y = (row_idx + 1) * row_height
            draw.line([(0, sep_y), (frame_width, sep_y)], fill='gray', width=1)

    return frame


# Build frame data: 3 starters per frame, generated molecules chunked with cumulative index
allFrameData = []
starterList = list(starterToGeneratedMap.items())

for starter_smi, gen_list in starterList:
    genChunks = [gen_list[i:i + 2] for i in range(0, len(gen_list), 3)]
    startIdx = 0
    for chunk in genChunks:
        allFrameData.append((starter_smi, chunk, startIdx))
        startIdx += len(chunk)

# Group into frames of 3 starters each
framesGrouped = [allFrameData[i:i + 3] for i in range(0, len(allFrameData), 3)]

print(f"Total frames available: {len(framesGrouped)}")

# Evenly sample 1000 frames
maxFrames = 100
if len(framesGrouped) > maxFrames:
    sampleIndices = np.linspace(0, len(framesGrouped) - 1, maxFrames, dtype=int)
    framesSampled = [framesGrouped[i] for i in sampleIndices]
else:
    framesSampled = framesGrouped

print(f"Frames to generate: {len(framesSampled)}")

# Create sampled frames
allFrameImages = []
for frame_idx, frameGroup in enumerate(framesSampled):
    frameImg = create_frame(frameGroup)
    allFrameImages.append(frameImg)
    if (frame_idx + 1) % 100 == 0:
        print(f"Generated frame {frame_idx + 1}/{len(framesSampled)}")

# Save as GIF
gifOutputPath = os.path.join(doranetOutputDir, f"{fileNamePrefix}.gif")

allFrameImages[0].save(
    gifOutputPath,
    save_all=True,
    append_images=allFrameImages[1:],
    duration=3000,
    loop=0
)

gifOutputPath = os.path.join(doranetOutputDir, f"{fileNamePrefix}.gif")
print(f"Total frames: {len(allFrameImages)}")

Step 1 - Removed 304 helper molecules
Remaining molecules: 28049 (98.93%)
Step 2 - Removed 528 duplicate molecular SMILES (98.12%)
Final unique molecules: 27521 (97.07% of original)

Starters with generated molecules: 2
Total frames available: 147
Frames to generate: 100
Generated frame 100/100
Total frames: 100


### Drug-Likeness Analysis of molecules present in `uniqueMolecules`

Drug-likeness is a qualitative assessment used in drug discovery to determine whether a molecule has physicochemical and structural properties consistent with orally active drugs. Molecules that satisfy drug-likeness criteria are more likely to exhibit favorable **absorption, distribution, metabolism, excretion, and toxicity (ADMET)** profiles.

In this analysis, each unique molecule is evaluated against **five widely used drug-likeness filters**:

### 1. Lipinski's Rule of Five (Ro5)
Proposed by **Christopher Lipinski (1997)**, this is the most well-known drug-likeness filter. It predicts poor absorption or permeation when **more than one** of the following conditions is violated:

- Molecular Weight ≤ 500 Da
- LogP ≤ 5
- Hydrogen Bond Donors (HBD) ≤ 5
- Hydrogen Bond Acceptors (HBA) ≤ 10

> **Reference:** Lipinski, C.A. et al. *Adv. Drug Deliv. Rev.* **23**, 3–25 (1997). [DOI:10.1016/S0169-409X(96)00423-1](https://doi.org/10.1016/S0169-409X(96)00423-1)

### 2. Veber's Rules
Proposed by **Veber et al. (2002)**, these rules focus on oral bioavailability based on molecular flexibility and polar surface area:

- Number of Rotatable Bonds ≤ 10
- Topological Polar Surface Area (TPSA) ≤ 140 Å²

> **Reference:** Veber, D.F. et al. *J. Med. Chem.* **45**, 2615–2623 (2002). [DOI:10.1021/jm020017n](https://doi.org/10.1021/jm020017n)

### 3. Ghose Filter
Proposed by **Ghose et al. (1999)**, this filter defines a drug-like chemical space using ranges for key properties:

- 160 ≤ Molecular Weight ≤ 480 Da
- −0.4 ≤ LogP ≤ 5.6
- 20 ≤ Number of Heavy Atoms ≤ 70
- 40 ≤ Molar Refractivity ≤ 130

> **Reference:** Ghose, A.K. et al. *J. Comb. Chem.* **1**, 55–68 (1999). [DOI:10.1021/cc9800071](https://doi.org/10.1021/cc9800071)

### 4. Egan Filter
Proposed by **Egan et al. (2000)**, this filter predicts passive intestinal absorption using a simple two-parameter model:

- LogP ≤ 5.88
- TPSA ≤ 131.6 Å²

> **Reference:** Egan, W.J. et al. *J. Med. Chem.* **43**, 3867–3877 (2000). [DOI:10.1021/jm000292e](https://doi.org/10.1021/jm000292e)

### 5. Muegge Filter
Proposed by **Muegge et al. (2001)**, this pharmacophore-based filter uses a broader set of criteria to identify drug-like molecules:

- 200 ≤ Molecular Weight ≤ 600 Da
- −2 ≤ LogP ≤ 5
- TPSA ≤ 150 Å²
- Number of Rings ≤ 7
- Hydrogen Bond Acceptors ≤ 10
- Hydrogen Bond Donors ≤ 5
- Number of Rotatable Bonds ≤ 15
- Number of Heavy Atoms ≥ 8

> **Reference:** Muegge, I. et al. *J. Med. Chem.* **44**, 1841–1846 (2001). [DOI:10.1021/jm015507e](https://doi.org/10.1021/jm015507e)

### Drug-Like Score

Each molecule receives a **DrugLikeScore** ranging from **0 to 5**, representing the number of filters it passes. A score of **5** indicates the molecule satisfies all five drug-likeness criteria, suggesting strong potential as an orally bioavailable drug candidate.

| DrugLikeScore | Interpretation |
|:---:|:---|
| 5 | Excellent drug-likeness — passes all filters |
| 4 | Strong drug-likeness — minor deviation in one filter |
| 3 | Moderate drug-likeness |
| 2 | Weak drug-likeness |
| 1 | Poor drug-likeness |
| 0 | Not drug-like by any standard filter |

### Molecular Properties Computed

| Property | Description | Used By |
|:---|:---|:---|
| Molecular Weight (MW) | Exact molecular weight in Daltons | Lipinski, Ghose, Muegge |
| LogP | Octanol-water partition coefficient (lipophilicity) | Lipinski, Ghose, Egan, Muegge |
| HBD | Number of hydrogen bond donors | Lipinski, Muegge |
| HBA | Number of hydrogen bond acceptors | Lipinski, Muegge |
| TPSA | Topological polar surface area (Å²) | Veber, Egan, Muegge |
| Rotatable Bonds | Number of rotatable bonds | Veber, Muegge |
| Heavy Atoms | Number of non-hydrogen atoms | Ghose, Muegge |
| Molar Refractivity | Measure of molecular polarizability | Ghose |
| Rings | Total number of rings | Muegge |
| Aromatic Rings | Number of aromatic rings | Structural characterization |

> **Note:** These filters are designed primarily for **orally active small molecules**. Natural products and biologics may legitimately violate some criteria while still being therapeutically relevant. The DrugLikeScore should be interpreted as a guide, not an absolute cutoff.

In [24]:
from rdkit.Chem import Descriptors, rdMolDescriptors, Lipinski


# Drug-likeness filter functions
def computeDrugLikeProperties(smiles):
    """Compute drug-likeness properties from SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return pd.Series({
            'MolWeight': None,
            'LogP': None,
            'NumHBD': None,
            'NumHBA': None,
            'TPSA': None,
            'NumRotatableBonds': None,
            'NumRings': None,
            'NumAromaticRings': None,
            'NumHeavyAtoms': None,
            'MolarRefractivity': None,
            'PassesLipinski': False,
            'LipinskiViolations': None,
            'PassesVeber': False,
            'PassesGhose': False,
            'PassesEgan': False,
            'PassesMuegge': False,
            'DrugLikeScore': 0,
            'IsValidMol': False
        })

    molWeight = Descriptors.ExactMolWt(mol)
    logP = Descriptors.MolLogP(mol)
    numHBD = rdMolDescriptors.CalcNumHBD(mol)
    numHBA = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    numRotatableBonds = rdMolDescriptors.CalcNumRotatableBonds(mol)
    numRings = rdMolDescriptors.CalcNumRings(mol)
    numAromaticRings = rdMolDescriptors.CalcNumAromaticRings(mol)
    numHeavyAtoms = mol.GetNumHeavyAtoms()
    molarRefractivity = Descriptors.MolMR(mol)

    lipinskiViolations = 0
    if molWeight > 500:
        lipinskiViolations += 1
    if logP > 5:
        lipinskiViolations += 1
    if numHBD > 5:
        lipinskiViolations += 1
    if numHBA > 10:
        lipinskiViolations += 1
    passesLipinski = lipinskiViolations <= 1

    passesVeber = (numRotatableBonds <= 10) and (tpsa <= 140)

    passesGhose = (
        (160 <= molWeight <= 480) and
        (-0.4 <= logP <= 5.6) and
        (20 <= numHeavyAtoms <= 70) and
        (40 <= molarRefractivity <= 130)
    )

    passesEgan = (logP <= 5.88) and (tpsa <= 131.6)

    passesMuegge = (
        (200 <= molWeight <= 600) and
        (-2 <= logP <= 5) and
        (tpsa <= 150) and
        (numRings <= 7) and
        (numHBA <= 10) and
        (numHBD <= 5) and
        (numRotatableBonds <= 15) and
        (numHeavyAtoms >= 8)
    )

    drugLikeScore = sum([
        passesLipinski,
        passesVeber,
        passesGhose,
        passesEgan,
        passesMuegge
    ])

    return pd.Series({
        'MolWeight': round(molWeight, 4),
        'LogP': round(logP, 4),
        'NumHBD': numHBD,
        'NumHBA': numHBA,
        'TPSA': round(tpsa, 2),
        'NumRotatableBonds': numRotatableBonds,
        'NumRings': numRings,
        'NumAromaticRings': numAromaticRings,
        'NumHeavyAtoms': numHeavyAtoms,
        'MolarRefractivity': round(molarRefractivity, 4),
        'PassesLipinski': passesLipinski,
        'LipinskiViolations': lipinskiViolations,
        'PassesVeber': passesVeber,
        'PassesGhose': passesGhose,
        'PassesEgan': passesEgan,
        'PassesMuegge': passesMuegge,
        'DrugLikeScore': drugLikeScore,
        'IsValidMol': True
    })


# Compute drug-likeness for all molecules
print(f"Computing drug-likeness properties for {len(uniqueMolecules)} molecules...")

drugLikeProperties = uniqueMolecules['SMILES'].apply(computeDrugLikeProperties)

uniqueMoleculesDrug = pd.concat(
    [uniqueMolecules.reset_index(drop=True), drugLikeProperties],
    axis=1
)

invalidMolCount = len(uniqueMoleculesDrug[~uniqueMoleculesDrug['IsValidMol']])
print(f"Invalid molecules: {invalidMolCount}")
print(f"Final DataFrame shape: {uniqueMoleculesDrug.shape}")

# Summary
validMols = uniqueMoleculesDrug[uniqueMoleculesDrug['IsValidMol']]
totalValid = len(validMols)

lipinskiCount = validMols['PassesLipinski'].sum()
veberCount = validMols['PassesVeber'].sum()
ghoseCount = validMols['PassesGhose'].sum()
eganCount = validMols['PassesEgan'].sum()
mueggeCount = validMols['PassesMuegge'].sum()
allFiveCount = len(validMols[validMols['DrugLikeScore'] == 5])
nonePassedCount = len(validMols[validMols['DrugLikeScore'] == 0])

print("\nDrug likeliness summary:")
print(f"Total valid molecules: {totalValid}")
print(f"  Passes Lipinski (Ro5):  {lipinskiCount} ({lipinskiCount/totalValid*100:.2f}%)")
print(f"  Passes Veber:           {veberCount} ({veberCount/totalValid*100:.2f}%)")
print(f"  Passes Ghose:           {ghoseCount} ({ghoseCount/totalValid*100:.2f}%)")
print(f"  Passes Egan:            {eganCount} ({eganCount/totalValid*100:.2f}%)")
print(f"  Passes Muegge:          {mueggeCount} ({mueggeCount/totalValid*100:.2f}%)")
print(f"\n  Passes ALL 5 filters:   {allFiveCount} ({allFiveCount/totalValid*100:.2f}%)")
print(f"  Passes NONE:            {nonePassedCount} ({nonePassedCount/totalValid*100:.2f}%)")

# DrugLikeScore distribution
print(f"\nDrugLikeScore Distribution:")
for score in range(6):
    count = len(validMols[validMols['DrugLikeScore'] == score])
    print(f"  Score {score}/5: {count} ({count/totalValid*100:.2f}%)")

# Breakdown: starters vs generated
startersDrug = validMols[validMols['Is_Starter'] == True]
generatedDrug = validMols[validMols['Is_Starter'] == False]

starterAllFive = len(startersDrug[startersDrug['DrugLikeScore'] == 5])
generatedAllFive = len(generatedDrug[generatedDrug['DrugLikeScore'] == 5])

starterPercent = (starterAllFive / len(startersDrug) * 100) if len(startersDrug) > 0 else 0
generatedPercent = (generatedAllFive / len(generatedDrug) * 100) if len(generatedDrug) > 0 else 0

print(f"\nAmong starters:  {starterAllFive}/{len(startersDrug)} pass all 5 ({starterPercent:.2f}%)")
print(f"Among generated: {generatedAllFive}/{len(generatedDrug)} pass all 5 ({generatedPercent:.2f}%)")

uniqueMoleculesDrug

Computing drug-likeness properties for 27521 molecules...
Invalid molecules: 0
Final DataFrame shape: (27521, 21)

Drug likeliness summary:
Total valid molecules: 27521
  Passes Lipinski (Ro5):  12995 (47.22%)
  Passes Veber:           11005 (39.99%)
  Passes Ghose:           6244 (22.69%)
  Passes Egan:            7947 (28.88%)
  Passes Muegge:          7882 (28.64%)

  Passes ALL 5 filters:   4110 (14.93%)
  Passes NONE:            11834 (43.00%)

DrugLikeScore Distribution:
  Score 0/5: 11834 (43.00%)
  Score 1/5: 4809 (17.47%)
  Score 2/5: 2288 (8.31%)
  Score 3/5: 1782 (6.48%)
  Score 4/5: 2698 (9.80%)
  Score 5/5: 4110 (14.93%)

Among starters:  0/2 pass all 5 (0.00%)
Among generated: 4110/27519 pass all 5 (14.94%)


,SMILES,Is_Starter,SourceDirectories,MolWeight,LogP,NumHBD,NumHBA,TPSA,NumRotatableBonds,NumRings,...,NumHeavyAtoms,MolarRefractivity,PassesLipinski,LipinskiViolations,PassesVeber,PassesGhose,PassesEgan,PassesMuegge,DrugLikeScore,IsValidMol
0,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,False,starter_00005;starter_00007;starter_00014;star...,506.9957,-1.6290,7,14,279.13,8,3,...,31,95.4757,False,3,False,False,False,False,0,True
1,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)O)[C@@H]...,False,starter_00005;starter_00007;starter_00014;star...,347.0631,-1.8630,5,10,186.07,4,3,...,23,73.6551,True,0,False,False,False,False,1,True
2,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)...,False,starter_00005;starter_00007;starter_00014;star...,785.1571,-2.4240,9,21,362.93,13,6,...,53,176.8337,False,3,False,False,False,False,0,True
3,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,False,starter_00005;starter_00007;starter_00014;star...,787.1728,-1.7503,11,20,363.28,13,6,...,53,180.2071,False,3,False,False,False,False,0,True
4,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,False,starter_00005;starter_00007;starter_00014;star...,427.0294,-1.7460,6,12,232.60,6,3,...,27,84.5654,False,2,False,False,False,False,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27516,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,starter_00073,1394.2011,6.7463,3,31,442.44,24,12,...,95,313.3074,False,3,False,False,False,False,0,True
27517,Cn1nnc(-c2ccc(-c3ccc(OP(=O)(OC[C@H]4CN(c5ccc(-...,False,starter_00073,1135.2364,6.5493,1,25,319.84,18,11,...,80,268.5478,False,3,False,False,False,False,0,True
27518,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,starter_00073,629.0837,1.7624,3,14,226.65,11,5,...,42,135.9493,False,2,False,False,False,False,0,True
27519,Cn1nnc(-c2ccc(-c3ccc(N4C[C@H](COP(=O)(O)OP(=O)...,False,starter_00073,1746.3095,9.1876,2,39,517.47,30,16,...,121,402.0786,False,3,False,False,False,False,0,True


### Check matches for Natural Product from public data bases via `SMILES`
 - NPAtlas: https://www.npatlas.org/download
 - COCONUT: https://coconut.naturalproducts.net/download

## Natural Product Matching via `InChIKey`

# How many pathways has been generated

In [21]:
import PyPDF2

# Automatically find all starter directories (exclude files)
allStarterDirPaths = sorted([
    p for p in glob.glob(os.path.join(doranetOutputDir, "starter_*"))
    if os.path.isdir(p)
])

totalStarterDirs = len(allStarterDirPaths)
print(f"Found {totalStarterDirs} starter molecules")

# Count pathways from PDF files in each directory
pathwayCountsPerStarter = []

for starterDirPath in allStarterDirPaths:
    dirName = os.path.basename(starterDirPath)
    starterNum = int(dirName.replace('starter_', ''))

    expectedPdfPath = os.path.join(starterDirPath, f"{dirName}_pathways_visualized.pdf")

    if not os.path.exists(expectedPdfPath):
        fallbackPdfPaths = glob.glob(os.path.join(starterDirPath, "*_pathways_visualized.pdf"))
        expectedPdfPath = fallbackPdfPaths[0] if fallbackPdfPaths else None

    numPathways = 0
    if expectedPdfPath and os.path.exists(expectedPdfPath):
        try:
            with open(expectedPdfPath, 'rb') as pdfFile:
                pdfReader = PyPDF2.PdfReader(pdfFile)
                numPathways = len(pdfReader.pages)
        except Exception as e:
            print(f"Error reading PDF for {dirName}: {e}")

    pathwayCountsPerStarter.append({
        'StarterDirectory': dirName,
        'StarterNum': starterNum,
        'NumPathways': numPathways
    })

allPathwaysRaw = pd.DataFrame(pathwayCountsPerStarter).sort_values('StarterNum').reset_index(drop=True)

# Keep only starters with pathways
allPathways = allPathwaysRaw[allPathwaysRaw['NumPathways'] > 0].copy().reset_index(drop=True)

totalPathways = allPathways['NumPathways'].sum()
startersWithPathways = len(allPathways)
startersWithoutPathways = totalStarterDirs - startersWithPathways
withPercent = (startersWithPathways / totalStarterDirs) * 100
withoutPercent = (startersWithoutPathways / totalStarterDirs) * 100

print(f"Total pathways found: {totalPathways}")
print(f"Starters with pathways: {startersWithPathways} ({withPercent:.2f}%), "
      f"without: {startersWithoutPathways} ({withoutPercent:.2f}%)")

allPathways = allPathways.drop(columns=['StarterNum'])
allPathways

Found 354 starter molecules
Total pathways found: 137
Starters with pathways: 72 (20.34%), without: 282 (79.66%)


,StarterDirectory,NumPathways
0,starter_00000,1
1,starter_00001,1
2,starter_00003,2
3,starter_00004,2
4,starter_00005,1
...,...,...
67,starter_00347,4
68,starter_00348,4
69,starter_00350,1
70,starter_00351,4


### Visualize pathways

### Discover All Starter Directories

In [ ]:
doranetOutputDir_NP = "doranet_output_NP"

helpersToExclude = {
    'O', 'O=O', '[H][H]', 'O=C=O', 'C=O', '[C-]#[O+]', 'Br', '[Br][Br]',
    'CO', 'C=C', 'O=S(O)O', 'N', 'O=S(=O)(O)O', 'O=NO', 'N#N',
    'O=[N+]([O-])O', 'NO', 'C#N', 'S', 'O=S=O', 'N#CO', '[H+]', 'OO',
    'Cl', 'I', 'O=C(O)O', 'O=P(O)(O)O', 'O=P(O)(O)OP(=O)(O)O', 'C',
    'CC', 'CC=O', 'CC(=O)O', 'CCC(=O)O'
}

print(f"Output directory: {doranetOutputDir_NP}")
print(f"Number of helpers to exclude: {len(helpersToExclude)}")

In [ ]:
# Automatically find all starter directories
allStarterDirPaths_NP = sorted(glob.glob(os.path.join(doranetOutputDir_NP, "starter_*_NP")))

print(f"Found {len(allStarterDirPaths_NP)} starter directories")

# Find the molecules CSV file inside each directory
discoveredCsvFiles = []
directoriesWithMissingCsv = []

for starterDirPath in allStarterDirPaths_NP:
    dirName = os.path.basename(starterDirPath)

    # Expected CSV file name matches directory name
    expectedCsvPath = os.path.join(starterDirPath, f"{dirName}_molecules.csv")

    if os.path.exists(expectedCsvPath):
        discoveredCsvFiles.append({
            'dirName': dirName,
            'dirPath': starterDirPath,
            'csvPath': expectedCsvPath,
            'starterNum': int(dirName.replace('starter_', '').replace('_NP', ''))
        })
    else:
        # Try to find any molecules CSV in the directory as fallback
        fallbackCsvPaths = glob.glob(os.path.join(starterDirPath, "*_molecules.csv"))
        if fallbackCsvPaths:
            discoveredCsvFiles.append({
                'dirName': dirName,
                'dirPath': starterDirPath,
                'csvPath': fallbackCsvPaths[0],
                'starterNum': int(dirName.replace('starter_', '').replace('_NP', ''))
            })
        else:
            directoriesWithMissingCsv.append(dirName)

# Sort by starter number
discoveredCsvFiles.sort(key=lambda x: x['starterNum'])

print(f"Found {len(discoveredCsvFiles)} CSV files containing DORAnet generated molecules")
if directoriesWithMissingCsv:
    print(f"Missing CSV in {len(directoriesWithMissingCsv)} directories: "
          f"{directoriesWithMissingCsv[:10]}...")

### Read All CSV Files into a DataFrame

In [ ]:
# Read all CSV files and combine into one DataFrame
perStarterDataFrames = []
csvReadErrors = []

for fileInfo in discoveredCsvFiles:
    try:
        singleStarterMolecules = pd.read_csv(fileInfo['csvPath'])

        # Add source starter information
        singleStarterMolecules['SourceStarterNum'] = fileInfo['starterNum']
        singleStarterMolecules['SourceDirectory'] = fileInfo['dirName']

        perStarterDataFrames.append(singleStarterMolecules)

    except Exception as e:
        csvReadErrors.append({
            'dirName': fileInfo['dirName'],
            'csvPath': fileInfo['csvPath'],
            'error': str(e)
        })

# Concatenate all DataFrames
if perStarterDataFrames:
    allMoleculesWithDuplicates_wNP = pd.concat(perStarterDataFrames, ignore_index=True)
    print(f"Total rows read (with duplicates and helpers): {len(allMoleculesWithDuplicates_wNP)}")
    print(f"Columns: {list(allMoleculesWithDuplicates_wNP.columns)}")
    print(f"Unique SMILES (before cleaning): {allMoleculesWithDuplicates_wNP['SMILES'].nunique()}")
else:
    print("ERROR: No CSV files could be read!")

if csvReadErrors:
    print(f"\nFailed to read {len(csvReadErrors)} files:")
    for err in csvReadErrors[:5]:
        print(f"  {err['dirName']}: {err['error']}")

allMoleculesWithDuplicates_wNP

### Remove HELPERS and Deduplicate

In [ ]:
# Step 1: Remove helpers
allMoleculesWithoutHelpers = allMoleculesWithDuplicates_wNP[
    ~allMoleculesWithDuplicates_wNP['SMILES'].isin(helpersToExclude)
].copy()

helpersRemovedCount = len(allMoleculesWithDuplicates_wNP) - len(allMoleculesWithoutHelpers)
helpersKeptPercent = (len(allMoleculesWithoutHelpers) / len(allMoleculesWithDuplicates_wNP)) * 100
print(f"Step 1 - Removed {helpersRemovedCount} helper molecules")
print(f"Remaining molecules: {len(allMoleculesWithoutHelpers)} ({helpersKeptPercent:.2f}%)")

# Step 2: Aggregate source starters per unique SMILES
sourceStarterMapping = (
    allMoleculesWithoutHelpers
    .groupby('SMILES')['SourceStarterNum']
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)
sourceStarterMapping.columns = ['SMILES', 'SourceStarterList']
sourceStarterMapping['NumSourceStarters'] = sourceStarterMapping['SourceStarterList'].apply(len)
sourceStarterMapping['SourceStarters'] = sourceStarterMapping['SourceStarterList'].apply(
    lambda x: ';'.join(map(str, x))
)

# Also aggregate source directories per unique SMILES
sourceDirectoryMapping = (
    allMoleculesWithoutHelpers
    .groupby('SMILES')['SourceDirectory']
    .apply(lambda x: ';'.join(sorted(set(x))))
    .reset_index()
)
sourceDirectoryMapping.columns = ['SMILES', 'SourceDirectories']

# Step 3: Deduplicate - keep first occurrence of each SMILES
uniqueMolecules_wNP = allMoleculesWithoutHelpers.drop_duplicates(
    subset='SMILES', keep='first'
).copy()

# Keep only desired columns
uniqueMolecules_wNP = uniqueMolecules_wNP[['SMILES', 'Is_Starter']].copy()

# Step 4: Merge aggregated source starter and directory information
uniqueMolecules_wNP = uniqueMolecules_wNP.merge(
    sourceStarterMapping[['SMILES', 'NumSourceStarters', 'SourceStarters']],
    on='SMILES',
    how='left'
)
uniqueMolecules_wNP = uniqueMolecules_wNP.merge(
    sourceDirectoryMapping,
    on='SMILES',
    how='left'
)

duplicatesRemovedCount = len(allMoleculesWithoutHelpers) - len(uniqueMolecules_wNP)
dedupeKeptPercent = (len(uniqueMolecules_wNP) / len(allMoleculesWithoutHelpers)) * 100
overallKeptPercent = (len(uniqueMolecules_wNP) / len(allMoleculesWithDuplicates_wNP)) * 100

print(f"Step 2 - Removed {duplicatesRemovedCount} duplicate molecular SMILES ({dedupeKeptPercent:.2f}%)")
print(f"Final unique molecules: {len(uniqueMolecules_wNP)} ({overallKeptPercent:.2f}% of original)")

uniqueMolecules_wNP = uniqueMolecules_wNP.drop(columns=['NumSourceStarters', 'SourceStarters'])
uniqueMolecules_wNP

### How many pathways has been generated 

In [ ]:
import PyPDF2

# Count pathways from PDF files in each directory
pathwayCountsPerStarter = []

for starterDirPath in allStarterDirPaths_NP:
    dirName = os.path.basename(starterDirPath)
    starterNum = int(dirName.replace('starter_', '').replace('_NP', ''))

    expectedPdfPath = os.path.join(starterDirPath, f"{dirName}_pathways_visualized.pdf")

    if not os.path.exists(expectedPdfPath):
        fallbackPdfPaths = glob.glob(os.path.join(starterDirPath, "*_pathways_visualized.pdf"))
        expectedPdfPath = fallbackPdfPaths[0] if fallbackPdfPaths else None

    numPathways = 0
    if expectedPdfPath and os.path.exists(expectedPdfPath):
        try:
            with open(expectedPdfPath, 'rb') as pdfFile:
                pdfReader = PyPDF2.PdfReader(pdfFile)
                numPathways = len(pdfReader.pages)
        except Exception as e:
            print(f"Error reading PDF for {dirName}: {e}")

    pathwayCountsPerStarter.append({
        'StarterDirectory': dirName,
        'StarterNum': starterNum,
        'NumPathways': numPathways
    })

allPathways_wNPRaw = pd.DataFrame(pathwayCountsPerStarter).sort_values('StarterNum').reset_index(drop=True)

# Keep only starters with pathways
allPathways_wNP = allPathways_wNPRaw[allPathways_wNPRaw['NumPathways'] > 0].copy().reset_index(drop=True)

totalStarterDirs_NP = len(allStarterDirPaths_NP)
totalPathways = allPathways_wNP['NumPathways'].sum()
startersWithPathways = len(allPathways_wNP)
startersWithoutPathways = totalStarterDirs_NP - startersWithPathways
withPercent = (startersWithPathways / totalStarterDirs_NP) * 100
withoutPercent = (startersWithoutPathways / totalStarterDirs_NP) * 100

print(f"Total pathways found: {totalPathways}")
print(f"Starters with pathways: {startersWithPathways} ({withPercent:.2f}%), "
      f"without: {startersWithoutPathways} ({withoutPercent:.2f}%)")

allPathways_wNP = allPathways_wNP.drop(columns=['StarterNum'])
allPathways_wNP